# 2.2 Guided Activity: Exploring PDB Structures in Python

In [**2.1 Molecular Dynamics I**](01-md-intro-and-force-fields.md), we saw that almost every molecular dynamics simulation starts from a **Protein Data Bank (`.pdb`)** file — a plain-text list of every atom's identity and 3D $(X, Y, Z)$ coordinates.

In this interactive Python activity, you will:
1. Inspect raw `ATOM` records inside a real experimental protein structure (**Villin**, `villin_water.pdb`).
2. Compare atom counts, residue lengths, and starting shapes across our four starter `.pdb` files using **MDAnalysis**.
3. Rotate and explore **Villin** and **extended polyalanine** (`polyALA.pdb`) in an interactive **3D molecular viewer**.
4. **Build a custom 3D peptide spelling your own name** using the 1-letter amino acid code!

Click **Activate Live Editable Python Cells** above to edit and run cells live — or scroll through in Reading Mode (even the 3D protein viewers below are already interactive!).

## Step 1: Load Python Libraries & Locate Starter PDB Files

We import `MDAnalysis` (for parsing protein structures and trajectories), `numpy`, and our Mitchell Lab `md_helpers` module (which provides `view_structure` and `build_name_peptide`).

In [1]:
import os
import sys
import warnings

warnings.simplefilter("ignore")

# Locate the Molecular Dynamics workspace folder (works in live containers and local checkouts)
for candidate in ("/home/jovyan/work/MD", "MD", "tutorials/MD", "../../tutorials/MD"):
    if os.path.isdir(candidate):
        os.chdir(candidate)
        break

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import MDAnalysis as mda
import numpy as np
from md_helpers import view_structure, build_name_peptide

print("MD workspace ready:", os.path.basename(os.getcwd()))
print("Available starter PDB files:", sorted(os.listdir("data")))

MD workspace ready: MD
Available starter PDB files: ['polyALA.pdb', 'polyGLY.pdb', 'polyGV.pdb', 'villin_water.pdb']


## Step 2: What Does a `.pdb` File Look Like Inside?

Let's open `data/villin_water.pdb` — the actin-binding protein **Villin** solvated in water — and print its first 15 lines. Match each column back to the anatomy diagram from **2.1**:

- `ATOM` record → atom serial → atom name (`N`, `CA`, `C`, `O`…) → residue (`LEU`, `SER`…) → residue # → **$X, Y, Z$ coordinates (Å)** → element.

In [2]:
pdb0_file = "data/villin_water.pdb"

with open(pdb0_file, "r") as fp:
    for i, line in enumerate(fp):
        if i < 15:
            print(line.rstrip())
        else:
            break

REMARK    GENERATED BY TRJCONV
HEADER    Villin N68H in explicit water
REMARK    THIS IS A SIMULATION BOX
CRYST1   49.163   45.981   38.869  90.00  90.00  90.00 P 1           1
MODEL        0
ATOM      1  N   LEU     1      25.160  14.160  19.440  1.00  0.00
ATOM      2  H1  LEU     1      24.350  13.730  19.870  1.00  0.00
ATOM      3  H2  LEU     1      25.980  13.680  19.760  1.00  0.00
ATOM      4  H3  LEU     1      25.180  15.100  19.810  1.00  0.00
ATOM      5  CA  LEU     1      25.090  13.920  17.980  1.00  0.00
ATOM      6  HA  LEU     1      24.700  12.930  17.760  1.00  0.00
ATOM      7  CB  LEU     1      24.420  14.970  17.100  1.00  0.00
ATOM      8  HB1 LEU     1      24.590  14.630  16.080  1.00  0.00
ATOM      9  HB2 LEU     1      24.950  15.910  17.240  1.00  0.00
ATOM     10  CG  LEU     1      22.930  15.010  17.400  1.00  0.00


## Step 3: Compare Our Starter Structures Quantitatively

Using `MDAnalysis.Universe`, we can immediately inspect how many **protein atoms**, **water atoms**, and **amino acid residues** each `.pdb` file contains — plus measure the **starting end-to-end distance** between the first amino acid's backbone nitrogen (`N`) and the last amino acid's backbone carbon (`C`).

In [3]:
starter_files = [
    ("villin_water.pdb", "Folded Villin headpiece in water"),
    ("polyALA.pdb", "25-residue extended polyalanine (Ala)₂₅"),
    ("polyGLY.pdb", "25-residue extended polyglycine (Gly)₂₅"),
    ("polyGV.pdb", "Extended Gly-Val alternating peptide"),
]

print(f"{'File':<18} {'Residues':>8} {'Prot Atoms':>11} {'Water Atoms':>12} {'End-to-End (Å)':>15}  Description")
print("-" * 102)
for fname, desc in starter_files:
    u = mda.Universe(f"data/{fname}")
    prot = u.select_atoms("protein")
    water = u.select_atoms("resname HOH WAT TIP3 SOL")
    n_res = len(prot.residues)
    n_term = prot.select_atoms(f"resid {prot.residues[0].resid} and name N")
    c_term = prot.select_atoms(f"resid {prot.residues[-1].resid} and name C")
    dist = float(np.linalg.norm(n_term.positions[0] - c_term.positions[0]))
    print(f"{fname:<18} {n_res:>8d} {len(prot):>11d} {len(water):>12d} {dist:>14.1f}  {desc}")

File               Residues  Prot Atoms  Water Atoms  End-to-End (Å)  Description
------------------------------------------------------------------------------------------------------


villin_water.pdb         35         582         8283            8.5  Folded Villin headpiece in water
polyALA.pdb              25         253            0           85.3  25-residue extended polyalanine (Ala)₂₅
polyGLY.pdb              25         178            0           85.3  25-residue extended polyglycine (Gly)₂₅
polyGV.pdb               29         332            0           99.1  Extended Gly-Val alternating peptide


## Step 4: Explore Villin in 3D

Drag to rotate, scroll to zoom, and use the toolbar buttons to switch between **Cartoon**, **Cartoon + Sticks**, **Sticks**, and **Spheres** — or colour by **Secondary Structure** to highlight Villin's three packed $\alpha$-helices!

*(Tip: Notice how compactly folded 35-residue Villin is — only ~8.5 Å end-to-end — compared to our 85 Å stretched `polyALA.pdb` chain below! Want to see all the surrounding water molecules too? Change `show_water=False` to `show_water=True` and re-run!)*

In [4]:
view_structure(
    "data/villin_water.pdb",
    style="cartoon",
    color="spectrum",
    show_water=False,
    title="Villin Headpiece (villin_water.pdb — 35 residues)",
)

## Step 5: Inspect Extended Polyalanine (`polyALA.pdb`)

Next week we will simulate this fully extended 25-residue **polyalanine** chain (`polyALA.pdb`) in OpenMM and watch it spontaneously fold into an $\alpha$-helix!

Try switching to `data/polyGLY.pdb` below to compare how **Glycine** lacks Alanine's methyl ($-\text{CH}_3$) sidechains.

In [5]:
view_structure(
    "data/polyALA.pdb",
    style="cartoon+stick",
    color="element",
    title="Extended Polyalanine (polyALA.pdb — 25 Ala residues)",
)

## Step 6: Build a 3D Peptide Spelling Your Own Name!

Every amino acid has a **1-letter code** (`A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y`) — leaving out 6 letters of the English alphabet (`B, J, O, U, X, Z`).

Change `MY_NAME` below to **your own name** (or a friend's name, or a word like `"HELICASE"` or `"CANCER"`) and run the cell! `build_name_peptide(...)` will:
1. Map each valid letter to its amino acid and print its chemical properties.
2. Construct a 3D extended polypeptide chain and add force-field-ready hydrogens (`amber10.xml`) so you can even **simulate it folding in Week 3**!
3. Display your personal peptide in interactive 3D!

In [6]:
MY_NAME = "MITCHELL"   # <-- Change this to your own name and press Shift + Enter!

name_pdb, seq, n_res = build_name_peptide(MY_NAME)
view_structure(
    name_pdb,
    style="cartoon+stick",
    color="element",
    title=f"Custom Peptide for '{MY_NAME}' → {seq} ({n_res} residues)",
)

Input name        : MITCHELL
Peptide sequence  : MITCHELL (8 residues, 134 atoms)
Saved PDB file    : data/peptide_MITCHELL.pdb

Residue breakdown:
   1. M (MET) — Methionine     [Hydrophobic (sulphur-containing)]
   2. I (ILE) — Isoleucine     [Hydrophobic (branched)]
   3. T (THR) — Threonine      [Polar (hydroxyl group)]
   4. C (CYS) — Cysteine       [Polar (forms disulphide bonds)]
   5. H (HIS) — Histidine      [Polar / positively charged]
   6. E (GLU) — Glutamate      [Negatively charged (acidic)]
   7. L (LEU) — Leucine        [Hydrophobic (α-helix former)]
   8. L (LEU) — Leucine        [Hydrophobic (α-helix former)]


---

### Ready for Week 3?

Now that you can read, quantify, visualise, and even **build** `.pdb` structures in Python, head to **Week 3** to learn about **energy landscapes & trajectory analysis** in [**3.1 Molecular Dynamics II**](03-energy-landscapes-and-analysis.md) and run a live **OpenMM protein folding simulation** in [**3.2 Live OpenMM Practical**](04-protein-folding-simulation.ipynb)!